# Занятие 7, пара 2. Трекинг и классические детекторы


Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai


**Как проходить тетрадь.** Основные примеры работают без ответов «Ваш ход».
        Если пропускаете упражнение, пропустите также его `check(...)`. Продолжайте следующий обычный раздел.
        Сохраняйте свои попытки в отдельной копии. Проверки помогают разобраться, но не выставляют баллы.
        Интернет, Git и регистрация для основной работы не нужны.


## Рамка на кадре — ещё не подсчёт объектов

Детектор отвечает «где сейчас кандидат на объект?». Трекер связывает наблюдения во времени:
«это тот же объект?». Счётчик добавляет правило события: например, центр пересёк линию слева направо.
Одна посылка видна на десятках кадров, но от этого на складе не появляется несколько десятков посылок.

Построим понятную цепочку для одного объекта, сравним два варианта на одном ролике и разберём,
почему хорошие рамки не гарантируют правильный счёт. Затем посмотрим, чем каскад и HOG отличаются
от нашего порогового детектора. Нужны маски, расстояния и IoU из предыдущих блоков; данные загрузим заново.
Основной путь — NumPy/Matplotlib. Результат — воспроизводимый счёт с анализом ошибок, таблица сравнения
и три задачи на ограничения метода. В конце выдаётся ЛР3 по прежним критериям и 15 баллам.


### Подключим учебные файлы

        Следующая ячейка — готовая служебная настройка. Она ищет `course_support` в рабочей папке
        или выше. Её не надо переписывать по памяти. `DATA_DIR` указывает на учебные данные,
        `WORKSPACE_DIR` — на корень вашей копии, `check` запускает проверку упражнения.
        Если поддержка не найдена, перенесите весь комплект, а не только notebook.
        В этой тетради создаются все необходимые переменные: другую тетрадь запускать не нужно.


In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check, summary

PROJECT_DIR


### 1. Сначала посчитаем глазами

Посмотрите короткий предпросмотр и отметьте, сколько раз центр посылки пересёк вертикальную линию x=320
**слева направо**. Линия — правило задачи; в исходные пиксели она не нарисована.
Запишите ручной результат до чтения вывода программы. Остановка сама по себе не создаёт новый объект.
Рамки `boxes` из файла — точная геометрия генератора: они нужны позже для оценки, не для обнаружения.


In [ ]:
import json
import time
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import Image, display
from course_support.video import component_boxes

clip = np.load(DATA_DIR / 'video' / 'conveyor.npz', allow_pickle=False)
frames = clip['frames']
background_rgb = clip['background']
line_x = int(clip['line_x'])
reference_boxes = clip['boxes']
display(Image(filename=str(DATA_DIR / 'video' / 'preview.gif')))
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for axis, index in zip(axes, [29,32,40]):
    axis.imshow(frames[index])
    axis.axvline(line_x,color='red')
    axis.set_title('Кадр ' + str(index))
plt.show()


### 2. Из маски — в кандидаты на объекты

Фоновая модель из 07.1 выделяет пиксели. Чтобы получить рамку, объединим соседние белые пиксели
в 4-связные области. Это не распознавание посылки: соединённая тень тоже могла бы стать областью.

`component_boxes` — готовый служебный обход маски, исходник в course_support/video.py.
Вход — bool-маска, `min_area` — минимальная площадь **в пикселях переданной маски**.
Выход — список (x0,y0,x1,y1,area). Правая и нижняя границы не включены: ширина x1−x0.
Сначала проверим смысл на небольшой маске, не изучая заново циклы Python.


In [ ]:
small_mask = np.zeros((8,12),dtype=bool)
small_mask[1:4,1:4] = True
small_mask[5:7,8:11] = True
small_mask[0,11] = True
print(component_boxes(small_mask,min_area=3))


Для CPU возьмём каждый четвёртый пиксель по обеим осям. Маска станет 90×160.
Площадь для min_area относится к этой маленькой сетке. Вернув рамку, умножим координаты на 4;
на границах возникнет квантование до нескольких пикселей, поэтому IoU не обязан быть ровно 1.

Функция ниже соединяет разобранные шаги: среднее RGB → разность с пустым фоном → при необходимости
вычитание медианного сдвига света → порог → компоненты → крупнейшая область.
Выбираем крупнейшую потому, что в этом опыте ведём **одну** посылку; несколько посылок потребуют иной логики.


In [ ]:
def detect_box(frame, reference, compensate):
    current = frame[::4,::4].astype(np.float32).mean(axis=2)
    empty = reference[::4,::4].astype(np.float32).mean(axis=2)
    difference = current - empty
    if compensate:
        difference = difference - np.median(difference)
    mask = np.abs(difference) > 25
    components = component_boxes(mask,min_area=20)
    if len(components)==0:
        return None
    areas = []
    for component in components:
        areas.append(component[4])
    selected = components[int(np.argmax(areas))]
    x0,y0,x1,y1,area = selected
    return (4*x0,4*y0,4*x1,4*y1)

first_box = detect_box(frames[0],background_rgb,True)
print('Найденная рамка:',first_box)


`Rectangle((x0,y0), width,height)` рисует рамку на Matplotlib-осях.
Не перепутайте её ширину/высоту с координатами правого нижнего угла.
Покажем найденный результат: эталон генератора в вычислении first_box не участвовал.


In [ ]:
fig, axis = plt.subplots(figsize=(11,6))
axis.imshow(frames[0])
if first_box is not None:
    x0,y0,x1,y1 = first_box
    axis.add_patch(Rectangle((x0,y0),x1-x0,y1-y0,fill=False,edgecolor='lime',linewidth=2))
axis.axvline(line_x,color='red')
plt.show()


### 3. Когда считать, что это тот же объект?

Начинаем с первой найденной рамки. На следующем кадре измеряем расстояние между центрами.
Если центр прыгнул дальше 24 пикселей за кадр, отвергаем соответствие: наш объект так быстро
не двигался в выбранной постановке. Это параметр модели движения; изменение fps меняет его смысл.

При отсутствии рамки или слишком большом скачке помечаем трек потерянным. В этой простой цепочке
он **не восстанавливается автоматически**: так мы не выдаём другой объект за прежний.
Нет признаков внешности, прогноза скорости и обработки перекрытий — это реальные ограничения.
Детектор всё равно запускаем на всех кадрах, чтобы границы сравнения времени были одинаковы.


In [ ]:
def box_center(box):
    x0,y0,x1,y1 = box
    return np.array([(x0+x1)/2,(y0+y1)/2])

def track_one(sequence, reference, compensate, max_jump=24):
    predictions = []
    previous = None
    lost = False
    for frame in sequence:
        detected = detect_box(frame,reference,compensate)
        current = detected
        if not lost and current is not None and previous is not None:
            distance = np.linalg.norm(box_center(current)-box_center(previous))
            if distance>max_jump:
                current = None
        if current is None:
            lost = True
        if lost:
            current = None
        predictions.append(current)
        previous = current
    return predictions

fixed_tracks = track_one(frames,background_rgb,False)
aligned_tracks = track_one(frames,background_rgb,True)


Теперь определим событие, а не количество кадров. Для непрерывных наблюдений пересечение вправо:
предыдущее x < line, текущее x >= line. При пропуске сбрасываем предыдущее наблюдение: не приписываем
неизвестному промежутку уверенное пересечение. Это простой счёт **событий**, ещё не защита от дрожания.
`None` означает, что координата неизвестна, а не равна нулю.


In [ ]:
def track_x(boxes):
    values = []
    for box in boxes:
        values.append(None if box is None else float(box_center(box)[0]))
    return values

def count_rightward(values,line):
    total = 0
    previous = None
    for current in values:
        if current is not None and previous is not None:
            if previous<line and current>=line:
                total += 1
        previous = current
    return total

print('Неизменный фон:',count_rightward(track_x(fixed_tracks),line_x))
print('С поправкой света:',count_rightward(track_x(aligned_tracks),line_x))
fig, axis = plt.subplots(figsize=(11,4))
axis.plot(track_x(fixed_tracks),marker='o',markersize=4,linewidth=0,label='Неизменный фон')
axis.plot(track_x(aligned_tracks),label='Поправка света')
axis.axhline(line_x,color='red',label='Линия счёта')
axis.axvline(30,color='gray',linestyle=':',label='Изменение света')
axis.set(xlabel='Номер кадра',ylabel='x центра, px')
axis.legend()
plt.show()


### 4. Сравним качество и затраты честно

Счёт сравниваем с собственным визуальным подсчётом: в учебной сцене одно пересечение вправо.
Для геометрии используем рамки генератора. IoU рамок — площадь пересечения / площадь объединения,
как у масок, только площади прямоугольников считаются по координатам. Пропущенная рамка получает 0;
не выбрасываем трудные кадры из среднего. Время измеряем отдельно, без отрисовки и чтения файла.
`time.perf_counter()` — часы для коротких интервалов. Один прогрев и медиана трёх повторов
уменьшают случайные колебания; они не превращают ноутбук в универсальный аппаратный тест.


In [ ]:
def box_iou(a,b):
    if a is None or b is None:
        return 0.0
    width = max(0,min(a[2],b[2])-max(a[0],b[0]))
    height = max(0,min(a[3],b[3])-max(a[1],b[1]))
    intersection = width*height
    area_a = (a[2]-a[0])*(a[3]-a[1])
    area_b = (b[2]-b[0])*(b[3]-b[1])
    return intersection/(area_a+area_b-intersection)

comparison = []
for name,compensate in [('fixed',False),('aligned',True)]:
    track_one(frames[:2],background_rgb,compensate)
    durations = []
    for repeat in range(3):
        start = time.perf_counter()
        predictions = track_one(frames,background_rgb,compensate)
        durations.append(time.perf_counter()-start)
    scores = []
    for predicted,truth in zip(predictions,reference_boxes):
        scores.append(box_iou(predicted,truth))
    count = count_rightward(track_x(predictions),line_x)
    row = {'method':name,'count':count,'count_error':abs(count-1),
           'mean_iou':float(np.mean(scores)),'seconds':float(np.median(durations))}
    comparison.append(row)
    print(row)


Разница результата связана с предположением о свете, а не с «лучшим названием библиотеки».
На большей части кадра baseline ошибся после смены освещения; компенсация сохранила объект.
На контрпримере из 07.1 могла бы проиграть и она. Наше время включает уменьшение, маску,
компоненты и сопровождение на всех кадрах; не включает декодирование/показ. Для реальной камеры
нужна отдельная оценка полной задержки. Теперь проверим, может ли даже высокий IoU обмануть счётчик.


### ✏️ Ваш ход — хорошие рамки, плохой счёт

Создайте два ряда центров x длиной от 4 до 10: `my_truth_x` и `my_predicted_x`.
Истинный ряд один раз пересекает линию 320 вправо, предсказанный — два раза из-за дрожания.
Рамки имеют ширину и высоту 80, центр y=200. Придумайте небольшие ошибки так, чтобы IoU
каждой пары рамок был больше 0.9. Сохраните ряды, постройте график и объясните: почему хорошая
средняя геометрическая метрика не гарантирует правильный ответ бизнес-задачи?


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('07.2.1', my_truth_x, my_predicted_x)


**Мой вывод:** …


### 5. Состояние, зона нечувствительности и ID

Введём полосу [line−margin, line+margin]. Левее неё состояние «слева», правее — «справа»,
внутри — неопределённая сторона. Пока центр внутри полосы, сохраняем последнюю уверенную сторону.
Событие подтверждается переходом слева направо. Это гистерезис: результат зависит от предыстории.
Вместо дрожания на одном пикселе объект должен пройти всю полосу. За устойчивость платим задержкой.

Если считаем уникальные объекты, запоминаем уже учтённые ID. Пропуск координаты None сбрасывает
сторону, но не память о том, что объект уже посчитан. После пропуска не знаем, где он пересёк линию.
ID ниже предоставлены во входе; задача не реализует автоматическое присваивание ID нескольким объектам.

Для состояния удобно использовать словарь по ID. `sides.get(object_id)` вернёт сохранённую сторону или None, если ID ещё не встречался. Множество уже посчитанных ID хранится отдельно; повторное добавление того же ID не увеличивает его размер.


### ✏️ Ваш ход — счёт с памятью и неизвестностью

Напишите `my_count_ids(records,line,margin)`: число разных ID,
для которых подтверждён переход из левой уверенной стороны в правую. records — упорядоченные
наблюдения `(object_id, x)`, x может быть None. Внутри полосы сторону не меняем; после None
предыдущую сторону забываем; один ID учитываем не более раза. Границы полосы относятся к ней.
Проверьте: дрожание только внутри полосы; два разных ID; появление сразу справа; пропуск перед
появлением справа. Объясните, почему смена ID у того же физического предмета всё ещё может испортить счёт.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('07.2.2', my_count_ids)


**Мой вывод:** …


### 6. А если нужен именно класс объекта?

Наш детектор знает «что отличается от фона», но не знает слово «лицо» или «человек».
Классические обучаемые детекторы сравнивают признаки окна с закономерностями, найденными на примерах.

**Haar-признак** — разность сумм яркости в прямоугольных областях. На одинаковой площади он
замечает, например, тёмную полосу над светлой. Одно такое число не доказывает наличие лица.
Каскад последовательно отвергает неподходящие окна простыми проверками; пороги и сочетания признаков
получены при обучении. В OpenCV мы загружаем готовую модель, не обучаем её одним вызовом detectMultiScale.


In [ ]:
toy_window = np.full((8,12),180,dtype=float)
toy_window[2:4] = 60
dark_sum = toy_window[2:4].sum()
light_sum = toy_window[4:6].sum()
print('Один прямоугольный признак:',light_sum-dark_sum)

portrait = np.load(DATA_DIR / 'video' / 'portrait.npy',allow_pickle=False)
with (DATA_DIR / 'video' / 'cascade_example.json').open(encoding='utf-8') as file:
    prepared = json.load(file)
fig, axis = plt.subplots(figsize=(8,8))
axis.imshow(portrait)
for x,y,width,height in prepared['boxes_xywh']:
    axis.add_patch(Rectangle((x,y),width,height,fill=False,edgecolor='lime',linewidth=2))
axis.set_title('Заранее рассчитанная иллюстрация Haar cascade')
plt.show()


Это реальная фотография NASA из открытого набора; происхождение в video/SOURCE.md.
Рамки здесь **заранее рассчитаны при подготовке**, не найдены запущенным вами детектором.
При наличии OpenCV можно повторить расчёт в необязательной ячейке ниже. Нулевой результат допустим:
он означает, что этот детектор не выдал рамок при данных условиях, а не доказывает отсутствие лиц.

**HOG** описывает направления и силу перепадов яркости в маленьких ячейках изображения.
В отличие от цветовой гистограммы, он описывает локальные края. Нормировка групп соседних ячеек
уменьшает зависимость от контраста; затем, например, линейный SVM оценивает окно по этому вектору.
Полноценный HOG сохраняет расположение ячеек и их нормировку. Один общий столбчатый график
направлений ниже — иллюстрация идеи, не весь HOG и не обученный детектор людей.


Резерв: `np.gradient` оценивает производные по осям; для массива изображения возвращает gy, затем gx.
`np.hypot(gx,gy)` — длина вектора градиента, `arctan2(gy,gx)` — угол, degrees переводит его в градусы.
Остаток от деления на 180 объединяет противоположные направления: считаем ориентацию края без знака.
В `np.histogram` параметр weights означает, что сильный градиент вносит больший вклад, чем слабый.


In [ ]:
portrait_gray = portrait.astype(float).mean(axis=2)
gy,gx = np.gradient(portrait_gray)
strength = np.hypot(gx,gy)
orientation = np.degrees(np.arctan2(gy,gx)) % 180
hist,edges = np.histogram(orientation,bins=9,range=(0,180),weights=strength)
fig, axes = plt.subplots(1, 2, figsize=(12,5))
axes[0].imshow(strength,cmap='gray')
axes[0].set_title('Сила градиента')
axes[1].bar(edges[:-1],hist,width=20,align='edge')
axes[1].set(xlabel='Ориентация, градусы',ylabel='Сумма сил градиента')
plt.show()


### 7. Рамки есть — значит, всё закрыли?

Детектор может пропустить объект, выдать лишнюю рамку или обрезать нужную область.
Для демонстрации закроем выданные прямоугольники сплошной заливкой в **копии** фото.
Это редактирование предоставленных областей. Оно не подтверждает правильность обнаружения
или обезличивание файла: остаются пропуски, контекст и другие признаки. Тест «рамок достаточно много»
тем более не проверяет, какие именно пиксели закрыты.


In [ ]:
covered_portrait = portrait.copy()
for x,y,width,height in prepared['boxes_xywh']:
    covered_portrait[y:y+height,x:x+width] = 0
plt.figure(figsize=(8,8))
plt.imshow(covered_portrait)
plt.title('Закрашены только предоставленные области')
plt.show()


### ✏️ Ваш ход — проверить покрытие, а не количество рамок

Проверяющий вручную указал две области, которые должны
быть целиком закрыты: (10,20,30,50), (80,20,110,50). Программа закрыла (10,20,30,50),
(80,20,100,50), (50,0,70,15). Все рамки — x0,y0,x1,y1, внутри изображения 90×150.
Напишите `my_coverage(shape,required_boxes,covered_boxes)`: доля закрытых пикселей для каждой
нужной области. Постройте общую bool-маску всех закрашенных рамок: перекрытия нельзя считать дважды.
Проверьте ещё случай, когда две частичные рамки вместе закрывают одну нужную область полностью.
Объясните, почему даже полный результат этого теста не проверяет полноту ручного списка объектов.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('07.2.3', my_coverage)


**Мой вывод:** …


### Дополнительно: собственный запуск каскада

В некоторых сборках OpenCV классического API нет. Проверяем и импорт, и наличие CascadeClassifier,
и загрузку XML. `detectMultiScale` ищет окна разных размеров: scaleFactor задаёт шаг масштаба,
minNeighbors влияет на подтверждение кандидатов. Это не вероятность и не гарантированная точность.
Для показанного кадра переводим RGB в GRAY; в VideoCapture исходные каналы были BGR.


In [ ]:
try:
    import cv2
except (ImportError,OSError):
    cv2 = None

detected_faces = None
if cv2 is None or not hasattr(cv2,'CascadeClassifier'):
    print('Каскадный API недоступен; выше была отдельно подписанная иллюстрация.')
elif not hasattr(cv2,'data') or not hasattr(cv2.data,'haarcascades'):
    print('В этой сборке нет каталога XML-моделей; используйте подписанную иллюстрацию.')
else:
    model_path = Path(cv2.data.haarcascades) / 'haarcascade_frontalface_default.xml'
    detector = cv2.CascadeClassifier(str(model_path))
    if detector.empty():
        print('Модель не загружена. Основная работа не зависит от XML.')
    else:
        detected_faces = detector.detectMultiScale(cv2.cvtColor(portrait,cv2.COLOR_RGB2GRAY),
                                                   scaleFactor=1.1,minNeighbors=5)
        fig, axis = plt.subplots(figsize=(8,8))
        axis.imshow(portrait)
        for x,y,width,height in detected_faces:
            axis.add_patch(Rectangle((x,y),width,height,fill=False,edgecolor='yellow',linewidth=2))
        axis.set_title('Результат вашего запуска каскада')
        plt.show()


### Результат и ЛР3

Сохраните сравнение двух методов, свой контрпример и вывод об устойчивости счёта.
[Условия ЛР3](../docs/ЛР3.md) в студенческой папке workbooks; из корня курса откройте docs/ЛР3.md.
Работа соединяет сопоставление/гомографию блока 05 с видео этой пары. Максимум — прежние 15 баллов,
срок объявляет преподаватель. Файлы или архив равноправны GitHub.

**Резерв 10–15 минут:** предложите, как восстановить потерянный трек и не посчитать тот же объект
повторно. Что будет, если две похожие посылки перекроются? Сначала схема и трудный пример;
писать полноценный многообъектный трекер за остаток пары не требуется.

По желанию: [каскад в OpenCV](https://docs.opencv.org/4.x/db/d28/tutorial_cascade_classifier.html),
[оригинальная работа Dalal–Triggs о HOG](https://lear.inrialpes.fr/people/triggs/pubs/Dalal-cvpr05.pdf).
CSRT/KCF/MIL — другие семейства готовых трекеров; их наличие зависит от сборки, а смысл проверки
потери объекта и ошибок от названия трекера не меняется.
